In [9]:
from google import genai
from google.genai import types
from datetime import datetime, timedelta

# ==========================================
# 1. ΤΑ ΕΡΓΑΛΕΙΑ ΤΟΥ AGENT (The Integrations)
# ==========================================

def calculate_quarantine_window(scan_time: str, freezer_time_mins: int, belt_time_mins: int = 5) -> str:
    """Υπολογίζει την ώρα που το μολυσμένο προϊόν βγαίνει από το τούνελ."""
    time_format = "%H:%M:%S"
    start_time = datetime.strptime(scan_time, time_format)
    total_mins = freezer_time_mins + belt_time_mins
    end_time = start_time + timedelta(minutes=total_mins)
    return f"Το παράθυρο μόλυνσης στην έξοδο ξεκινάει στις {end_time.strftime(time_format)}."

def search_boxes_in_google_sheets(target_time: str) -> str:
    """Ψάχνει στο (προσομοιωμένο) Google Sheet Παραγωγής για κιβώτια μετά την target_time."""
    # Εδώ στην κανονική εφαρμογή μπαίνει το google-sheets-api.
    # Για το demo, του επιστρέφουμε τα πραγματικά δεδομένα του Golden Path:
    return (
        "ΒΡΕΘΗΚΑΝ ΚΙΒΩΤΙΑ! "
        "Από τις 04:38:02 έως τις 04:41:15 παρήχθησαν κιβώτια (Lot: 09118-240726-0438 κ.λπ.). "
        "Όλα τοποθετήθηκαν στην ΠΑΛΕΤΑ με ID: LPN-260724-7153. "
        "ΕΝΕΡΓΕΙΑ: Η κατάσταση της παλέτας άλλαξε ψηφιακά σε 'BLOCKED'."
    )

def scan_google_drive_for_shipping(pallet_id: str) -> str:
    """Ψάχνει τα PDFs στο Google Drive για να δει αν η παλέτα φορτώθηκε."""
    # Εδώ μπαίνει το google-drive-api OCR.
    if pallet_id == "LPN-260724-7153":
        return (
            "ΚΡΙΣΙΜΟ - ΒΡΕΘΗΚΕ ΣΤΟ DRIVE! "
            "Η παλέτα LPN-260724-7153 υπάρχει στο παραστατικό '24072026FINAL.pdf'. "
            "Πελάτης: Μ. ΟΓΚΟΥΝΣΟΤΟ Μ.ΙΚΕ (Τσιμισκή 82, Θεσσαλονίκη). "
            "Όχημα Φόρτωσης: NBX7849. Κατάσταση: ΟΛΟΚΛΗΡΩΘΗΚΕ."
        )
    return "Δεν βρέθηκε η παλέτα στις αποστολές."

# ==========================================
# 2. ΣΤΗΣΙΜΟ ΤΟΥ AGENT & ΤΟΥ ΡΟΛΟΥ
# ==========================================

GOOGLE_API_KEY = "YOUR_API_KEY_HERE"
client = genai.Client(api_key=GOOGLE_API_KEY)

system_instruction = (
    "Είσαι ο Panthoron TraceAudit Agent. Είσαι Senior Quality Manager. "
    "Μόλις λάβεις μια ειδοποίηση μόλυνσης, ΠΡΕΠΕΙ ΝΑ ΚΑΝΕΙΣ ΤΑ ΕΞΗΣ ΒΗΜΑΤΑ ΜΕ ΤΗ ΣΕΙΡΑ: "
    "1. Να υπολογίσεις τον χρόνο καραντίνας καλώντας το 'calculate_quarantine_window'. "
    "2. Να ψάξεις τα κιβώτια στη βάση καλώντας το 'search_boxes_in_google_sheets'. "
    "3. Να σκανάρεις τα έγγραφα αποστολής καλώντας το 'scan_google_drive_for_shipping'. "
    "Στο τέλος, θέλω να τυπώσεις ένα ΕΠΙΣΗΜΟ URGENT RECALL REPORT (Αναφορά Ιχνηλασιμότητας) "
    "στα Αγγλικά, με bullets, απόλυτα δομημένο, για να σταλεί στη διοίκηση."
)

agent_config = types.GenerateContentConfig(
    system_instruction=system_instruction,
    tools=[calculate_quarantine_window, search_boxes_in_google_sheets, scan_google_drive_for_shipping],
    temperature=0.2
)

# ==========================================
# 3. ΤΟ ΚΡΙΣΙΜΟ EMAIL ΠΟΥ ΞΕΚΙΝΑΕΙ ΤΑ ΠΑΝΤΑ
# ==========================================

crisis_prompt = (
    "Επείγον: Η Α' Ύλη με Lot 260707AH βρέθηκε μολυσμένη. "
    "Από τον έλεγχο προκύπτει ότι έπεσε στη Γραμμή 1 στις 03:33:53. "
    "Το τελικό προϊόν (09118) έχει χρόνο τούνελ κατάψυξης 35 λεπτά. "
    "Ανέλαβε δράση, εντόπισε την παλέτα και βρες αν έχει φύγει σε πελάτη!"
)

print("Ο Agent αναλύει το περιστατικό και ψάχνει στις βάσεις δεδομένων...\n")

response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents=crisis_prompt,
    config=agent_config
)

print("="*60)
print(response.text)
print("="*60)

Ο Agent αναλύει το περιστατικό και ψάχνει στις βάσεις δεδομένων...

OFFICIAL URGENT RECALL REPORT

**Subject: Immediate Recall Required - Contaminated Product Identified and Shipped**

This report details an urgent product contamination incident requiring immediate action.

*   **Incident Overview:**
    *   Contaminated Raw Material Lot: 260707AH
    *   Contamination Point: Production Line 1
    *   Contamination Time (Raw Material Drop): 03:33:53
    *   Affected Final Product: 09118
    *   Freezer Tunnel Time: 35 minutes

*   **Contamination Window & Affected Product Identification:**
    *   Calculated Contamination Exit Window (from tunnel): Starting at 04:13:53
    *   Affected Production Batch: Boxes produced between 04:38:02 and 04:41:15
    *   Identified Pallet ID: LPN-260724-7153
    *   Current Pallet Status: Digitally changed to 'BLOCKED'

*   **Shipping Status:**
    *   **CRITICAL FINDING:** Pallet LPN-260724-7153 has been shipped.
    *   Shipping Document: '24072026F